# Clean: from raw API responses to a tidy dataset

Reads `data/raw/awards_raw.json` from `01_collect.ipynb` and writes one row per award per company to `data/awards_clean.csv`, plus a small sample for the repo.

In [1]:
import json

import pandas as pd

with open("data/raw/awards_raw.json") as f:
    raw = json.load(f)

awards = pd.DataFrame(raw)
print(awards.shape)
print(awards[["Award ID", "Recipient Name", "Award Amount", "NAICS", "PSC"]].head(3).to_string())

(18145, 15)
        Award ID               Recipient Name  Award Amount                                                        NAICS                                                                                                                         PSC
0  N0001917C0001  LOCKHEED MARTIN CORPORATION  3.513551e+10  {'code': '336411', 'description': 'AIRCRAFT MANUFACTURING'}                                                                     {'code': '1510', 'description': 'AIRCRAFT, FIXED WING'}
1  N0001902C3002  LOCKHEED MARTIN CORPORATION  3.417396e+10  {'code': '336411', 'description': 'AIRCRAFT MANUFACTURING'}  {'code': 'AC15', 'description': 'NATIONAL DEFENSE R&D SVCS; DEPARTMENT OF DEFENSE - MILITARY; R&D FACILITIES & MAJ EQUIP'}
2  N0001920C0009  LOCKHEED MARTIN CORPORATION  3.015475e+10  {'code': '336411', 'description': 'AIRCRAFT MANUFACTURING'}                                                                     {'code': '1510', 'description': 'AIRCRAFT, FIXED WING'}


## Flatten the nested fields and rename the columns

NAICS and PSC arrive as dictionaries, each with a code and a description.

In [2]:
awards["naics_code"] = awards["NAICS"].str["code"]
awards["naics_description"] = awards["NAICS"].str["description"]
awards["psc_code"] = awards["PSC"].str["code"]
awards["psc_description"] = awards["PSC"].str["description"]

awards = awards.rename(columns={
    "Award ID": "award_id",
    "generated_internal_id": "award_unique_id",
    "Recipient Name": "recipient_name",
    "Award Amount": "award_amount",
    "Start Date": "start_date",
    "End Date": "end_date",
    "Awarding Sub Agency": "awarding_branch",
    "Description": "description",
    "Contract Award Type": "contract_type",
})
print(awards.columns.tolist())

['internal_id', 'award_id', 'recipient_name', 'award_amount', 'start_date', 'end_date', 'awarding_branch', 'description', 'NAICS', 'PSC', 'contract_type', 'recipient_id', 'award_unique_id', 'company', 'fiscal_year_searched', 'naics_code', 'naics_description', 'psc_code', 'psc_description']


## Fix the types

In [3]:
awards["award_amount"] = pd.to_numeric(awards["award_amount"])
awards["start_date"] = pd.to_datetime(awards["start_date"], errors="coerce")
awards["end_date"] = pd.to_datetime(awards["end_date"], errors="coerce")
awards["description"] = awards["description"].str.strip()

print(awards[["award_amount", "start_date", "end_date", "description"]].dtypes)
print(awards[["award_amount", "start_date", "end_date", "description"]].isna().sum())
print(awards["award_amount"].describe().round(0))

award_amount           float64
start_date      datetime64[ns]
end_date        datetime64[ns]
description             object
dtype: object
award_amount      0
start_date        0
end_date          0
description     316
dtype: int64
count    1.814500e+04
mean     6.154521e+08
std      2.239122e+09
min      0.000000e+00
25%      6.000168e+07
50%      1.231530e+08
75%      3.508261e+08
max      3.573775e+10
Name: award_amount, dtype: float64


## Remove duplicates

Each search covers one fiscal year, and an award active in several years comes back from each of them. `Award ID` alone is not unique (orders under different parent contracts can share one), so `award_unique_id` is the key. A joint venture can belong to two companies, so an award can stay once per company.

In [4]:
before = len(awards)
awards = awards.drop_duplicates(subset=["company", "award_unique_id"])
print(f"{before:,} rows before, {len(awards):,} after, {before - len(awards):,} duplicates removed")
print(awards.groupby("company")["award_unique_id"].count().to_string())

18,145 rows before, 5,070 after, 13,075 duplicates removed
company
Anduril              255
Boeing               946
General Dynamics     929
Lockheed Martin      854
Northrop Grumman    1014
Palantir             129
RTX                  916
Scale AI              12
Shield AI             15


## Check the company names

The API's name search also returns subsidiaries and joint ventures listed under a parent. Every legal name below was checked by hand against the company it was filed under, and none are false matches, so no rows are dropped here.

In [5]:
print(awards.groupby("company")["recipient_name"].nunique().to_string())
print(awards.groupby("company")["recipient_name"].value_counts().groupby("company").head(3).to_string())

company
Anduril              2
Boeing              14
General Dynamics    30
Lockheed Martin     16
Northrop Grumman    12
Palantir             2
RTX                 11
Scale AI             1
Shield AI            1
company           recipient_name                               
Anduril           ANDURIL INDUSTRIES, INC.                         254
                  ANDURIL LLC                                        1
Boeing            THE BOEING COMPANY                               804
                  BELL BOEING JOINT PROJECT OFFICE                  91
                  BOEING SIKORSKY AIRCRAFT SUPPORT, LLC             12
General Dynamics  GENERAL DYNAMICS MISSION SYSTEMS, INC.           242
                  GENERAL DYNAMICS LAND SYSTEMS INC.               168
                  GENERAL DYNAMICS INFORMATION TECHNOLOGY, INC.    167
Lockheed Martin   LOCKHEED MARTIN CORPORATION                      666
                  LOCKHEED MARTIN CORP                             123
           

## Add fields for later analysis

The search returns any award active during a fiscal year, so some started before fiscal 2020. `award_amount` is the award's lifetime total, so those rows are flagged rather than dropped. Joint ventures can appear under two companies, which matters if dollars are ever added across companies.

In [6]:
awards["start_fiscal_year"] = awards["start_date"].dt.year + (awards["start_date"].dt.month >= 10).astype(int)
awards["started_before_fy2020"] = awards["start_fiscal_year"] < 2020
awards["shared_with_other_company"] = awards.duplicated(subset=["award_unique_id"], keep=False)

print(awards["started_before_fy2020"].value_counts().to_string())
print(awards["shared_with_other_company"].value_counts().to_string())
print("Awards with an amount of zero or less:", (awards["award_amount"] <= 0).sum())

started_before_fy2020
False    2689
True     2381
shared_with_other_company
False    5042
True       28
Awards with an amount of zero or less: 3


## Keep the final columns and save

In [7]:
columns = ["company", "award_id", "award_unique_id", "recipient_name", "recipient_id", "award_amount",
           "start_date", "end_date", "start_fiscal_year", "started_before_fy2020", "awarding_branch",
           "contract_type", "description", "naics_code", "naics_description", "psc_code", "psc_description",
           "shared_with_other_company"]

clean = awards[columns].sort_values(["company", "award_amount"], ascending=[True, False])
clean.to_csv("data/awards_clean.csv", index=False)

# The ten largest awards per company are small enough to commit
sample = clean.groupby("company").head(10)
sample.to_csv("data/sample_awards.csv", index=False)

print(clean.shape, sample.shape)
clean.info()
print(clean.isna().sum().to_string())
print(clean.head(3).to_string())

(5070, 18) (90, 18)
<class 'pandas.core.frame.DataFrame'>
Index: 5070 entries, 17795 to 18123
Data columns (total 18 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   company                    5070 non-null   object        
 1   award_id                   5070 non-null   object        
 2   award_unique_id            5070 non-null   object        
 3   recipient_name             5070 non-null   object        
 4   recipient_id               5070 non-null   object        
 5   award_amount               5070 non-null   float64       
 6   start_date                 5070 non-null   datetime64[ns]
 7   end_date                   5070 non-null   datetime64[ns]
 8   start_fiscal_year          5070 non-null   int64         
 9   started_before_fy2020      5070 non-null   bool          
 10  awarding_branch            5070 non-null   object        
 11  contract_type              5070 non-null   object